# 项目二：老用户激活与价值提升

## 业务背景

老用户 GMV 占比持续下滑，需要从「粗放式拉新」转向「精细化存量运营」。
本项目构建 **RFM-G 用户分层模型**，在传统 RFM 基础上增加品类拓展成长性（G）维度，
识别不同类型老用户的核心诉求，设计差异化运营策略。

**核心思路：**
1. **多维特征工程**：R（最近购买）、F（频次）、M（金额）、G（品类拓展）
2. **用户分层**：np.select 向量化分层（替代慢速 .apply）
3. **AB 测试论证**：卡方检验 + 双样本 z 检验，验证优惠策略效果
4. **分层运营**：针对不同层级的用户制定专属策略

**数据源：** `users` + `orders` + `order_items` + `products` + `user_activities`

---
## 1. 环境准备与数据获取

使用 SQL JOIN 一次性获取用户→订单→订单项→产品的宽表，
比分多次查询后在 pandas 中 merge 更高效。

In [ ]:
# ══════════════════════════════════════════════════════════
# 环境准备 & 数据库连接（支持 SQLite 和 MySQL 双模式）
# ══════════════════════════════════════════════════════════
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from datetime import datetime

sns.set_theme(style='whitegrid', context='talk', font='Microsoft YaHei')
plt.rcParams['axes.unicode_minus'] = False

# 改为 'mysql' 即可切换（前提: 先运行 migrate_to_mysql.py）
DB_TYPE = 'sqlite'

if DB_TYPE == 'sqlite':
    conn = sqlite3.connect('user_activation.db')
    # SQLite 中的表名
    T_USERS, T_ORDERS, T_ITEMS, T_PRODUCTS, T_ACTIVITIES = \
        'users', 'orders', 'order_items', 'products', 'user_activities'
    print('[SQLite] user_activation.db')
elif DB_TYPE == 'mysql':
    import pymysql
    conn = pymysql.connect(
        host='localhost', user='root', password='123456',
        database='ecommerce_analysis', charset='utf8mb4',
    )
    # MySQL 中的表名（与其他项目区分，加了 _p2 后缀）
    T_USERS, T_ORDERS, T_ITEMS, T_PRODUCTS, T_ACTIVITIES = \
        'users_p2', 'orders_p2', 'order_items_p2', 'products_p2', 'user_activities'
    print('[MySQL] ecommerce_analysis')

In [ ]:
# SQL JOIN 一次性获取五表宽表（表名根据 DB_TYPE 自动切换）
# 五表关系: T_USERS → T_ORDERS → T_ITEMS → T_PRODUCTS
query = f"""
    SELECT
        u.user_id, u.registration_date, u.gender, u.age,
        o.order_id, o.amount, o.order_status, o.create_time,
        oi.product_id, oi.quantity, oi.price,
        p.product_name, p.category_id
    FROM {T_USERS} u
    LEFT JOIN {T_ORDERS} o ON u.user_id = o.user_id
        AND o.order_status IN ('paid','shipped','completed')
    LEFT JOIN {T_ITEMS} oi ON o.order_id = oi.order_id
    LEFT JOIN {T_PRODUCTS} p ON oi.product_id = p.product_id
    ORDER BY u.user_id, o.create_time
"""
df_full = pd.read_sql(query, conn)
df_full['registration_date'] = pd.to_datetime(df_full['registration_date'])
df_full['create_time'] = pd.to_datetime(df_full['create_time'])

# 活动参与数据
activity_df = pd.read_sql(f'SELECT * FROM {T_ACTIVITIES}', conn)
activity_df['activity_date'] = pd.to_datetime(activity_df['activity_date'])

print(f'订单宽表: {len(df_full):,} 行')
print(f'活动数据: {len(activity_df):,} 行')
print(f'涉及用户: {df_full["user_id"].nunique():,} 人')
print(f'\n品类分布:')
print(df_full['category_id'].value_counts().rename({1: '袜子', 2: '服装'}))

---
## 2. RFM-G 特征工程

### G 维度（品类拓展成长性）的设计思路

传统 RFM 模型只衡量用户「买了多少」「花了多少钱」，
但无法反映用户是否在拓展新的消费品类——这对老用户激活至关重要。

**G 的计算：**
- `category_diversity`：购买过的品类数
- `growth_ratio`：服装品类订单占比（关注从袜子向服装的迁移）
- `growth`（综合）= 品类多样性 × (1 + 服装金额占比)

这样设计的原因是：一个用户如果从只买袜子转变为也买服装，
说明品牌粘性在增强，未来消费潜力更大。

In [ ]:
# ============================================================
# RFM-G 特征工程：从五表宽表聚合用户级指标
# ============================================================
# RFM-G = RFM + Growth（品类拓展成长性）
# G 维度的业务含义: 用户是否从"只买袜子"拓展到"也买服装"
#   品类拓展意味着品牌粘性增强，未来消费潜力更大

now = pd.Timestamp.now()

rfm = (
    # dropna(subset=['order_id']): 去掉没有订单的记录（LEFT JOIN 产生的 NULL 行）
    df_full.dropna(subset=['order_id'])
    .groupby('user_id')
    .agg(
        # ---- 传统 RFM 三维 ----
        # max(): 取最近一次购买时间
        last_purchase=('create_time', 'max'),
        # nunique(): 不重复的订单号计数
        frequency=('order_id', 'nunique'),
        # sum(): 所有订单金额加总
        monetary=('amount', 'sum'),

        # ---- G 维度：品类拓展 ----
        # nunique(): 买过几个品类（最多 2: 袜子+服装）
        category_diversity=('category_id', 'nunique'),
        # lambda: 统计每个用户有多少个服装品类(category_id=2)的订单
        # (x == 2).sum() → 把 True/False 转成 1/0 再求和
        category_2_orders=('category_id', lambda x: (x == 2).sum()),
        category_1_orders=('category_id', lambda x: (x == 1).sum()),
        # 服装品类的消费金额: 从 df_full 中筛选 category_id==2 的行再求和
        # x.index 是该用户在原表中所有行的索引，用它去 df_full 中查 category_id
        category_2_amount=('amount', lambda x: x[df_full.loc[x.index, 'category_id'] == 2].sum()),
    )
    .reset_index()
)

# Recency: 当前时间 - 最后一次购买时间 = 距今多少天
rfm['recency'] = (now - rfm['last_purchase']).dt.days

# ---- 综合成长性 G 的计算 ----
# growth_ratio: 服装订单占比（0~1，越大说明越倾向服装）
# +1e-6 防止除以 0（如果两个品类订单都是 0）
rfm['growth_ratio'] = rfm['category_2_orders'] / (rfm['category_1_orders'] + rfm['category_2_orders'] + 1e-6)
# growth_amount_pct: 服装金额占总消费的比例
rfm['growth_amount_pct'] = rfm['category_2_amount'] / (rfm['monetary'] + 1e-6)
# 综合 G = 品类多样性 × (1 + 服装金额占比)
# 逻辑: 买过跨品类的 + 服装花了更多钱的 → G 更高
rfm['growth'] = rfm['category_diversity'] * (1 + rfm['growth_amount_pct'])

# ---- 合并活动参与数据 ----
activity_summary = (
    activity_df.groupby('user_id')
    .agg(
        activity_count=('activity_id', 'nunique'),          # 参与过几种活动
        participation_count=('is_participated', 'sum'),     # 实际参与了几次
        participation_rate=('is_participated', 'mean'),     # 活动响应率
    )
    .reset_index()
)
# merge + fillna(0): 没参与过任何活动的用户填 0
rfm = rfm.merge(activity_summary, on='user_id', how='left').fillna(0)

print(f'有效用户数: {len(rfm)}')
print(f'品类拓展用户占比: {(rfm["category_2_orders"] > 0).mean():.1%}')
print(f'\nRFM-G 指标描述统计:')
print(rfm[['recency', 'frequency', 'monetary', 'growth', 'participation_rate']].describe().round(2))

---
## 3. RFM-G 评分与用户分层

In [ ]:
def safe_qcut(series, q, labels):
    """安全分箱"""
    actual_q = min(q, len(series.unique()))
    if actual_q < 2:
        return pd.Series([labels[-1]] * len(series), index=series.index)
    actual_labels = labels[-actual_q:] if len(labels) >= actual_q else labels
    try:
        result = pd.qcut(series, actual_q, labels=actual_labels, duplicates='drop')
    except ValueError:
        result = pd.cut(series, bins=actual_q, labels=actual_labels)
    return result


# 四维评分
rfm['r_score'] = safe_qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1])
rfm['f_score'] = safe_qcut(rfm['frequency'], 5, labels=[1, 2, 3, 4, 5])
rfm['m_score'] = safe_qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5])
rfm['g_score'] = safe_qcut(rfm['growth'], 5, labels=[1, 2, 3, 4, 5])

# np.select 向量化分层
score = rfm['r_score'].astype(int) + rfm['f_score'].astype(int) + rfm['m_score'].astype(int) + rfm['g_score'].astype(int)
conditions = [score >= 16, score >= 12, score >= 8]
choices = ['高价值深耕用户', '高潜唤醒用户', '成长型用户']
rfm['segment'] = np.select(conditions, choices, default='流失风险用户')

# 分层画像
segment_profile = (
    rfm.groupby('segment')
    .agg(
        user_count=('user_id', 'count'),
        avg_recency=('recency', 'mean'),
        avg_frequency=('frequency', 'mean'),
        avg_monetary=('monetary', 'mean'),
        avg_growth=('growth', 'mean'),
        avg_participation_rate=('participation_rate', 'mean'),
        avg_cat2_orders=('category_2_orders', 'mean'),
    )
    .round(2)
)

print('RFM-G 用户分层结果:')
for seg, row in segment_profile.iterrows():
    pct = row['user_count'] / len(rfm)
    print(f'\n  {seg} ({int(row["user_count"])}人, {pct:.1%})')
    print(f'    R={row["avg_recency"]:.0f}天  F={row["avg_frequency"]:.1f}次  '
          f'M={row["avg_monetary"]:.0f}元  G={row["avg_growth"]:.2f}')
    print(f'    活动参与率={row["avg_participation_rate"]:.1%}  服装订单={row["avg_cat2_orders"]:.1f}')

---
## 4. AB 测试分析

验证不同优惠策略对高潜唤醒用户的转化效果。

**统计方法：**
1. **卡方检验**：判断四组间转化率是否存在整体显著差异
2. **双样本比例 z 检验**：各实验组 vs 对照组的两两比较
3. **效应量（Lift）**：衡量实际提升幅度，而非仅看 p 值

> **实际工作中**，数据来自 CRM 实验系统的真实打点，
> 此处用模拟数据展示分析方法。

In [ ]:
# ============================================================
# AB 测试 — 用统计方法验证优惠策略是否有效
# ============================================================
# 模拟场景: 我们有 4 组用户，分别收到不同优惠策略
# 目标: 判断各组转化率的差异是真实效果还是随机波动
#
# 统计方法:
#   1. 卡方检验 (chi2_contingency): 整体检验四组间是否有显著差异
#      → 回答"至少有一组和别的不一样"
#   2. 双样本比例 z 检验: 逐一比较每组 vs 对照组
#      → 回答"具体哪组比对照组强"

np.random.seed(123)  # 固定随机种子，保证每次跑结果一样（可复现）
n_each = 200         # 每组 200 个样本

# 模拟各组的转化数据
# np.random.binomial(1, p, n): 抛 n 次硬币，每次正面概率为 p
#   返回 n 个 0/1 值，1 的比例约等于 p
groups = {
    '对照组 (无优惠)':    np.random.binomial(1, 0.08, n_each),  # 8% 转化率
    'A组 (10%折扣券)':   np.random.binomial(1, 0.14, n_each),  # 14% 转化率
    'B组 (满199减30)':   np.random.binomial(1, 0.18, n_each),  # 18% 转化率
    'C组 (买一送一)':    np.random.binomial(1, 0.22, n_each),  # 22% 转化率
}

# ---- 汇总各组的转化率 ----
ab_results = []
for name, conversions in groups.items():
    n = len(conversions)               # 该组总人数
    c = conversions.sum()              # 该组转化人数（0/1 求和 = 多少人是 1）
    ab_results.append({
        'group': name,
        'sample_size': n,
        'conversions': c,
        'conversion_rate': c / n,      # 转化率 = 转化人数 / 总人数
    })
ab_df = pd.DataFrame(ab_results)

print('各组转化率:')
for _, row in ab_df.iterrows():
    print(f'  {row["group"]:<20s}: {row["conversion_rate"]:.1%}  ({row["conversions"]}/{row["sample_size"]})')

# ---- 卡方检验 (Chi-Square Test) ----
# 检验原假设: "四组转化率没有差异"
# 如果 p < 0.05: 拒绝原假设，说明至少有一组和别的不一样
observed = [[r['conversions'], r['sample_size'] - r['conversions']] for r in ab_results]
chi2, p_value, dof, _ = stats.chi2_contingency(observed)
print(f'\n卡方检验: chi2={chi2:.2f}, df={dof}, p={p_value:.4f}')
print(f'结论: 各组转化率{"存在" if p_value < 0.05 else "无"}显著差异')

In [ ]:
# 两两比较（vs 对照组）
ctrl_rate = ab_results[0]['conversion_rate']
print('两两比较 (vs 对照组):')
print(f'{'实验组':<20s}  {'Lift':>8s}  {'p值':>8s}  {'显著性':>8s}')
print('-' * 50)

pairwise_results = []
for r in ab_results[1:]:
    n1, c1 = ab_results[0]['sample_size'], ab_results[0]['conversions']
    n2, c2 = r['sample_size'], r['conversions']
    # 双样本比例 z 检验
    p_pool = (c1 + c2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    z = (r['conversion_rate'] - ctrl_rate) / max(se, 1e-10)
    p_val = 2 * (1 - stats.norm.cdf(abs(z)))
    lift = (r['conversion_rate'] - ctrl_rate) / ctrl_rate
    significant = p_val < 0.05
    print(f'{r["group"]:<20s}  {lift:>+7.1%}  {p_val:>8.4f}  {"[显著]" if significant else "[不显著]":>8s}')
    pairwise_results.append({
        'comparison': f'对照组 vs {r["group"]}',
        'lift': lift, 'p_value': p_val, 'significant': significant,
    })

---
## 5. 综合可视化看板

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 14))
segments_ordered = ['高价值深耕用户', '高潜唤醒用户', '成长型用户', '流失风险用户']

# 图 1: 用户分层分布
ax = axes[0, 0]
seg_counts = rfm['segment'].value_counts()
colors_pie = ['#1565C0', '#42A5F5', '#90CAF9', '#E3F2FD']
ax.pie(seg_counts.values, labels=seg_counts.index, autopct='%1.1f%%',
       colors=colors_pie, explode=(0.05, 0.02, 0, 0))
ax.set_title('用户分层分布', fontweight='bold')

# 图 2: 各分层特征对比（归一化）
ax = axes[0, 1]
metrics = ['avg_frequency', 'avg_monetary', 'avg_growth', 'avg_participation_rate']
labels_cn = ['购买频次', '消费金额', '品类成长性', '活动参与率']
x = np.arange(len(metrics)); width = 0.2
for i, seg in enumerate(segments_ordered):
    if seg in segment_profile.index:
        vals = [segment_profile.loc[seg, m] for m in metrics]
        max_vals = segment_profile[metrics].max()
        # 用 .iloc[j] 按位置取值，因为 max_vals 的 index 是字符串不是数字
        vals_norm = [v / max(max_vals.iloc[j], 1) for j, v in enumerate(vals)]
        ax.bar(x + i * width, vals_norm, width, label=seg, alpha=0.85)
ax.set_xticks(x + width * 1.5); ax.set_xticklabels(labels_cn)
ax.set_title('各分层特征对比（归一化）', fontweight='bold')
ax.legend(fontsize=8)

# 图 3: 品类拓展散点图
ax = axes[0, 2]
for seg in segments_ordered:
    seg_data = rfm[rfm['segment'] == seg]
    ax.scatter(seg_data['category_1_orders'] + 0.5, seg_data['category_2_orders'] + 0.5,
               label=seg, alpha=0.6, s=40)
ax.set_xlabel('袜子品类订单数'); ax.set_ylabel('服装品类订单数')
ax.set_title('品类拓展散点图', fontweight='bold')
ax.legend(fontsize=8); ax.set_xscale('log'); ax.set_yscale('log')

# 图 4: Recency 分布
ax = axes[1, 0]
for seg in segments_ordered:
    seg_data = rfm[rfm['segment'] == seg]
    ax.hist(seg_data['recency'], bins=30, alpha=0.5, label=seg)
ax.set_xlabel('距上次购买天数'); ax.set_ylabel('用户数')
ax.set_title('Recency 分布 by 分层', fontweight='bold')
ax.legend(fontsize=8)

# 图 5: AB 测试转化率
ax = axes[1, 1]
bar_colors = ['#90CAF9', '#42A5F5', '#1E88E5', '#1565C0']
bars = ax.bar(ab_df['group'], ab_df['conversion_rate'], color=bar_colors, edgecolor='white')
ax.set_ylabel('转化率'); ax.set_title('AB 测试：各实验组转化率', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
for bar, rate in zip(bars, ab_df['conversion_rate']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{rate:.1%}', ha='center', fontweight='bold')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

# 图 6: GMV 恢复模拟
ax = axes[1, 2]
months = ['7月', '8月', '9月', '10月', '11月(双十一)', '12月']
gmv_before = [32, 30, 28, 26, 24, 23]
gmv_after  = [28, 29, 30, 31, 33, 34]
ax.plot(months, gmv_before, 'o-', color='red', linewidth=2, label='干预前（下滑趋势）')
ax.plot(months, gmv_after, 'o-', color='green', linewidth=2, label='干预后（回升趋势）')
ax.axvline(3.5, color='gray', linestyle='--', alpha=0.7, label='策略上线')
ax.set_ylabel('老用户 GMV 占比 (%)')
ax.set_title('老用户 GMV 占比趋势', fontweight='bold')
ax.legend(fontsize=8)

fig.suptitle('老用户激活与价值提升分析看板', fontsize=18, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('老用户激活与价值提升分析.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. 分层运营策略

| 分层 | 运营目标 | 核心策略 |
|------|---------|--------|
| 高价值深耕用户 | 提升客单价 & 品类拓展 | 专属 VIP 权益，高端服装线定向推荐 |
| 高潜唤醒用户 | 品类迁移（袜子→服装） | 品类拓展券，搭配购推荐，限时阶梯优惠 |
| 成长型用户 | 提升复购频次 | 品类教育内容，低门槛试用装 |
| 流失风险用户 | 召回唤醒 | 大力度召回券，社交裂变，简化购买流程 |

**预期效果：**
- 老用户复购率提升 6%
- 客单价提升 12%
- 老用户 GMV 占比回升 4-5 个百分点

In [ ]:
conn.close()
print('分析完成！')